# pRSAc MCMC Fitting
LLM 데이터 → pRSAc 파라미터 (φ_r, α, λ) 추정

| 모델 | 데이터 | Likelihood |
|------|--------|------------|
| A | speaker_logit  | Normal(log P_S1, σ) |
| B | speaker_choice | Multinomial(P_S1) |
| C | listener_logit | Normal(log P_L1, σ) |
| D | listener_choice| Multinomial(P_L1) |

**파라미터**
- `φ_r` : 관계별 epistemic weight (0=pure social, 1=pure epistemic)
- `α`   : social utility 스케일 (발화 선택에서 상태값이 얼마나 중요한지)
- `λ`   : rationality (높을수록 더 결정론적)

In [2]:
import re
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt
from pathlib import Path

from variables import ADJECTIVES, RELATIONSHIP_VAR, STATE_VAR

In [3]:
# ── CONFIG ──────────────────────── (여기만 수정)
MODEL = "llama3"    # "llama3" | "qwen3"
MCMC_DRAWS  = 2000
MCMC_TUNE   = 1000
MCMC_CHAINS = 4
# ────────────────────────────────────────────────

N_relations  = len(RELATIONSHIP_VAR)  # 4
N_utterances = len(ADJECTIVES)        # 5
N_states     = len(STATE_VAR)         # 5

print(f"Model: {MODEL}")
print(f"Relations : {RELATIONSHIP_VAR}")
print(f"Utterances: {ADJECTIVES}")
print(f"States    : {STATE_VAR}")

Model: llama3
Relations : ['Enge Freundin', 'Entfernte Kollegin', 'Lockere Chefin', 'Gefürchtete Chefin']
Utterances: ['großartig', 'gut', 'okay', 'schlecht', 'schrecklich']
States    : [1, 2, 3, 4, 5]


## 0. Semantics & L0 (Lumer Experiment 1)

In [4]:
# [[u]](s) : 발화 u가 상태 s에서 true일 확률 (Lumer et al.)
# row: utterance (großartig, gut, okay, schlecht, schrecklich)
# column: state (1, 2, 3, 4, 5)
semantics = np.array([
    [0.00000,0.03125,0.06250,0.15625,1.00000],  # großartig
    [0.03125,0.03125,0.65625,1.00000,0.90625],  # gut
    [0.06250,0.46875,0.93750,0.65625,0.46875],  # okay
    [0.90625,0.59375,0.06250,0.00000,0.00000],  # schlecht
    [0.84375,0.25000,0.00000,0.00000,0.00000]  # schrecklich
])  # shape: (N_utterances, N_states)

# L0: P_L0(s | u) ∝ [[u]](s) * P(s),  P(s) = uniform → will be removed at the end anyhow
sem_safe = np.where(semantics == 0, 1e-10, semantics)
# Nomarlisation 
L0 = sem_safe / sem_safe.sum(axis=1, keepdims=True)
# shape: (N_utterances, N_states),  L0[u, s] = P_L0(s | u)

states_arr = np.array(STATE_VAR, dtype=float)  # [1,2,3,4,5]

# U_epi(u, s) = log P_L0(s | u)  — 파라미터와 무관한 고정값
U_epi = np.log(L0)  # shape: (N_utterances, N_states)

# 각 행의 합이 1이 된다 
print("L0 (P_L0(s|u)):")
pd.DataFrame(L0, index=ADJECTIVES,
             columns=[f"s={s}" for s in STATE_VAR]).round(3)

L0 (P_L0(s|u)):


,s=1,s=2,s=3,s=4,s=5
großartig,0.000,0.024,0.048,0.128,0.800
gut,0.011,0.011,0.252,0.382,0.344
okay,0.023,0.181,0.362,0.254,0.181
schlecht,0.583,0.378,0.038,0.000,0.000
schrecklich,0.771,0.229,0.000,0.000,0.000


## 1. LLM 데이터 로드
`results/{role}_{mode}_{model}.csv` → numpy 텐서로 변환

In [ ]:
# parsing adjective from the speaker choice results
def _parse_adj(text):
    text = str(text).lower()
    found = {a: m.start() for a in ADJECTIVES
             if (m := re.search(rf"\b{a}\b", text))}
    return min(found, key=found.get) if found else None

#  listener choice 결과에서 숫자(1~5) 추출 + 글one, two로 썼을때는 해결안됌 
def _parse_state(text):
    m = re.search(r"[1-5]", str(text))
    return int(m.group()) if m else None


# 
def load_speaker_logit(model):
    """
    → logprobs_speaker[r, s, u]  = log P_LLM(u | state=s+1, rel=r)
       averaged over situations
    """
    p = Path(f"results/speaker_logit_{model}.csv")
    if not p.exists():
        print(f"[없음] {p}")
        return None
    df = pd.read_csv(p)
    out = np.zeros((N_relations, N_states, N_utterances))
    for r_i, rel in enumerate(RELATIONSHIP_VAR):
        for s_i, sta in enumerate(STATE_VAR):
            mask = (df["relationship"] == rel) & (df["state"] == sta)
            for u_i, adj in enumerate(ADJECTIVES):
                out[r_i, s_i, u_i] = df.loc[mask, f"logprob_{adj}"].mean()
    print(f"speaker_logit_{model}: shape {out.shape}")
    return out


def load_listener_logit(model):
    """
    → logprobs_listener[r, u, s]  = log P_LLM(s | adj=u, rel=r)
       averaged over situations
    """
    p = Path(f"results/listener_logit_{model}.csv")
    if not p.exists():
        print(f"[없음] {p}")
        return None
    df = pd.read_csv(p)
    out = np.zeros((N_relations, N_utterances, N_states))
    for r_i, rel in enumerate(RELATIONSHIP_VAR):
        for u_i, adj in enumerate(ADJECTIVES):
            mask = (df["relationship"] == rel) & (df["adjective"] == adj)
            for s_i, sta in enumerate(STATE_VAR):
                out[r_i, u_i, s_i] = df.loc[mask, f"logprob_{sta}"].mean()
    print(f"listener_logit_{model}: shape {out.shape}")
    return out


def load_speaker_counts(model):
    """
    → counts_speaker[r, s, u]  = count(u chosen | state=s+1, rel=r)
       aggregated over situations + repetitions
    """
    p = Path(f"results/speaker_choice_{model}.csv")
    if not p.exists():
        print(f"[없음] {p}")
        return None
    df = pd.read_csv(p)
    df["utterance"] = df["response_text"].apply(_parse_adj)
    df = df.dropna(subset=["utterance"])
    out = np.zeros((N_relations, N_states, N_utterances))
    for r_i, rel in enumerate(RELATIONSHIP_VAR):
        for s_i, sta in enumerate(STATE_VAR):
            mask = (df["relationship"] == rel) & (df["state"] == sta)
            for u_i, adj in enumerate(ADJECTIVES):
                out[r_i, s_i, u_i] = (df.loc[mask, "utterance"] == adj).sum()
    print(f"speaker_counts_{model}: shape {out.shape}, total={out.sum():.0f}")
    return out


def load_listener_counts(model):
    """
    → counts_listener[r, u, s]  = count(state s inferred | adj=u, rel=r)
       aggregated over situations + repetitions
    """
    p = Path(f"results/listener_choice_{model}.csv")
    if not p.exists():
        print(f"[없음] {p}")
        return None
    df = pd.read_csv(p)
    df["inferred"] = df["response_text"].apply(_parse_state)
    df = df.dropna(subset=["inferred"])
    df["inferred"] = df["inferred"].astype(int)
    out = np.zeros((N_relations, N_utterances, N_states))
    for r_i, rel in enumerate(RELATIONSHIP_VAR):
        for u_i, adj in enumerate(ADJECTIVES):
            mask = (df["relationship"] == rel) & (df["adjective"] == adj)
            for s_i, sta in enumerate(STATE_VAR):
                out[r_i, u_i, s_i] = (df.loc[mask, "inferred"] == sta).sum()
    print(f"listener_counts_{model}: shape {out.shape}, total={out.sum():.0f}")
    return out

In [ ]:
logprobs_speaker  = load_speaker_logit(MODEL)    # (4, 5, 5) or None
logprobs_listener = load_listener_logit(MODEL)   # (4, 5, 5) or None
counts_speaker    = load_speaker_counts(MODEL)   # (4, 5, 5) or None
counts_listener   = load_listener_counts(MODEL)  # (4, 5, 5) or None

## 2. pRSAc Forward Model

$$U(u,s,r) = \phi_r \cdot U_{epi}(u,s) + (1-\phi_r) \cdot U_{soc}(u)$$

$$P_{S1}(u|s,r) \propto \exp(\lambda \cdot U(u,s,r))$$

$$P_{L1}(s|u,r) \propto P(s) \cdot P_{S1}(u|s,r)$$

In [ ]:
def build_rsa(phi, alpha, lam):
    """
    pRSAc forward pass (pytensor).

    Parameters
    ----------
    phi   : (N_relations,)   — epistemic weight per relationship
    alpha : scalar           — social utility scale
    lam   : scalar           — rationality

    Returns
    -------
    log_S1 : (N_relations, N_utterances, N_states)
             log P_S1(u | s, r)
    log_L1 : (N_relations, N_utterances, N_states)
             log P_L1(s | u, r)
    """
    L0_t    = pt.as_tensor_variable(L0)         # (N_utt, N_states)
    U_epi_t = pt.as_tensor_variable(U_epi)      # (N_utt, N_states)
    s_t     = pt.as_tensor_variable(states_arr) # (N_states,)

    # U_soc(u) = α * Σ_s P_L0(s|u) * s
    U_soc = alpha * pt.dot(L0_t, s_t)           # (N_utt,)

    # 브로드캐스팅: → (N_rel, N_utt, N_states)
    phi_r    = phi[:, None, None]               # (N_rel, 1, 1)
    U_epi_b  = U_epi_t[None, :, :]             # (1, N_utt, N_states)
    U_soc_b  = U_soc[None, :, None]            # (1, N_utt, 1)

    U = phi_r * U_epi_b + (1 - phi_r) * U_soc_b  # (N_rel, N_utt, N_states)

    # S1: softmax over utterances (axis=1)
    logits_S1 = lam * U
    log_S1 = logits_S1 - pt.logsumexp(logits_S1, axis=1, keepdims=True)
    # log_S1[r, u, s] = log P_S1(u | s, r)  ✓

    # L1: Bayesian inversion, P(s) uniform → normalize S1 over states (axis=2)
    log_L1 = log_S1 - pt.logsumexp(log_S1, axis=2, keepdims=True)
    # log_L1[r, u, s] = log P_L1(s | u, r)  ✓

    return log_S1, log_L1

print("build_rsa 함수 정의 완료")

## Model A — Speaker Logit
`log P_LLM(u|s,r)` ~ Normal(`log P_S1(u|s,r)`, σ)

In [ ]:
trace_A = None

if logprobs_speaker is None:
    print("[skip] speaker_logit 데이터 없음")
else:
    # observed: (N_rel, N_states, N_utt) → (N_rel, N_utt, N_states)로 맞춤
    obs_A = logprobs_speaker.transpose(0, 2, 1)  # (N_rel, N_utt, N_states)

    with pm.Model() as model_A:
        phi   = pm.Uniform("phi",   lower=0, upper=1, shape=N_relations)
        alpha = pm.Uniform("alpha", lower=0, upper=5)
        lam   = pm.Uniform("lam",   lower=0, upper=20)
        sigma = pm.HalfNormal("sigma", sigma=1)

        log_S1, _ = build_rsa(phi, alpha, lam)

        pm.Normal("obs", mu=log_S1, sigma=sigma, observed=obs_A)

    with model_A:
        trace_A = pm.sample(
            draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS,
            target_accept=0.9, return_inferencedata=True,
        )

    az.plot_trace(trace_A, var_names=["phi", "alpha", "lam"])
    plt.tight_layout()
    plt.show()

## Model B — Speaker Choice (Freq)
count(u | s, r) ~ Multinomial(`P_S1(u|s,r)`)

In [ ]:
trace_B = None

if counts_speaker is None:
    print("[skip] speaker_choice 데이터 없음")
else:
    # counts_speaker[r, s, u],  n_sp[r, s] = trial 수
    n_sp = counts_speaker.sum(axis=-1)  # (N_rel, N_states)

    with pm.Model() as model_B:
        phi   = pm.Uniform("phi",   lower=0, upper=1, shape=N_relations)
        alpha = pm.Uniform("alpha", lower=0, upper=5)
        lam   = pm.Uniform("lam",   lower=0, upper=20)

        log_S1, _ = build_rsa(phi, alpha, lam)

        # P_S1[r, u, s] → transpose → [r, s, u] (counts_speaker 축 순서와 맞춤)
        P_S1 = pt.exp(log_S1).transpose(0, 2, 1)  # (N_rel, N_states, N_utt)

        pm.Multinomial("obs", n=n_sp, p=P_S1, observed=counts_speaker)

    with model_B:
        trace_B = pm.sample(
            draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS,
            target_accept=0.9, return_inferencedata=True,
        )

    az.plot_trace(trace_B, var_names=["phi", "alpha", "lam"])
    plt.tight_layout()
    plt.show()

## Model C — Listener Logit
`log P_LLM(s|u,r)` ~ Normal(`log P_L1(s|u,r)`, σ)

In [ ]:
trace_C = None

if logprobs_listener is None:
    print("[skip] listener_logit 데이터 없음")
else:
    # logprobs_listener[r, u, s] — 이미 올바른 축 순서
    with pm.Model() as model_C:
        phi   = pm.Uniform("phi",   lower=0, upper=1, shape=N_relations)
        alpha = pm.Uniform("alpha", lower=0, upper=5)
        lam   = pm.Uniform("lam",   lower=0, upper=20)
        sigma = pm.HalfNormal("sigma", sigma=1)

        _, log_L1 = build_rsa(phi, alpha, lam)

        pm.Normal("obs", mu=log_L1, sigma=sigma, observed=logprobs_listener)

    with model_C:
        trace_C = pm.sample(
            draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS,
            target_accept=0.9, return_inferencedata=True,
        )

    az.plot_trace(trace_C, var_names=["phi", "alpha", "lam"])
    plt.tight_layout()
    plt.show()

## Model D — Listener Choice (Freq)
count(s | u, r) ~ Multinomial(`P_L1(s|u,r)`)

In [ ]:
trace_D = None

if counts_listener is None:
    print("[skip] listener_choice 데이터 없음")
else:
    # counts_listener[r, u, s],  n_li[r, u] = trial 수
    n_li = counts_listener.sum(axis=-1)  # (N_rel, N_utt)

    with pm.Model() as model_D:
        phi   = pm.Uniform("phi",   lower=0, upper=1, shape=N_relations)
        alpha = pm.Uniform("alpha", lower=0, upper=5)
        lam   = pm.Uniform("lam",   lower=0, upper=20)

        _, log_L1 = build_rsa(phi, alpha, lam)

        P_L1 = pt.exp(log_L1)  # (N_rel, N_utt, N_states)

        pm.Multinomial("obs", n=n_li, p=P_L1, observed=counts_listener)

    with model_D:
        trace_D = pm.sample(
            draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS,
            target_accept=0.9, return_inferencedata=True,
        )

    az.plot_trace(trace_D, var_names=["phi", "alpha", "lam"])
    plt.tight_layout()
    plt.show()

## 결과 비교

In [ ]:
# ── 파라미터 요약 ──────────────────────────────────────────────────────────────
TRACES = {
    "A: Speaker Logit" : trace_A,
    "B: Speaker Choice": trace_B,
    "C: Listener Logit": trace_C,
    "D: Listener Choice": trace_D,
}

for name, trace in TRACES.items():
    if trace is None:
        continue
    print(f"\n{'='*55}")
    print(f" {name}")
    print('='*55)
    print(az.summary(trace, var_names=["phi", "alpha", "lam"], hdi_prob=0.95))

In [ ]:
# ── φ_r Cross-task 비교 플롯 ──────────────────────────────────────────────────
# Lumer 인간 데이터 기준값
LUMER_PHI = [0.45, 0.31, 0.37, 0.22]  # close, distant, easy-going, dreaded 순
# → RELATIONSHIP_VAR 순서: Enge Freundin, Entfernte Kollegin, Lockere Chefin, Gefürchtete Chefin

short_rel = ["Enge\nFreundin", "Entfernte\nKollegin",
             "Lockere\nChefin", "Gefürchtete\nChefin"]
x = np.arange(N_relations)
width = 0.18

fig, ax = plt.subplots(figsize=(10, 5))

colors = {"A": "#3498db", "B": "#2ecc71", "C": "#e74c3c", "D": "#f39c12"}
offsets = [-1.5, -0.5, 0.5, 1.5]

for i, (name, trace) in enumerate(TRACES.items()):
    if trace is None:
        continue
    key = name[0]
    phi_samples = trace.posterior["phi"].values  # (chains, draws, N_rel)
    phi_mean = phi_samples.mean(axis=(0, 1))     # (N_rel,)
    phi_std  = phi_samples.std(axis=(0, 1))
    ax.bar(x + offsets[i] * width, phi_mean, width,
           label=name, color=colors[key], alpha=0.8)
    ax.errorbar(x + offsets[i] * width, phi_mean, phi_std,
                fmt="none", color="black", capsize=3, linewidth=1)

# 인간 데이터
ax.plot(x, LUMER_PHI, "k--o", linewidth=2, markersize=7, label="Human (Lumer)")

ax.set_xticks(x)
ax.set_xticklabels(short_rel, fontsize=11)
ax.set_ylabel("φ_r  (epistemic weight)", fontsize=12)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.legend(fontsize=9)
ax.set_title(f"φ_r 추정값 비교 — {MODEL} vs Human (Lumer)", fontsize=13)
plt.tight_layout()
out = f"results/plot_phi_comparison_{MODEL}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장 → {out}")

In [ ]:
# ── α, λ 비교 표 ───────────────────────────────────────────────────────────────
rows = []
for name, trace in TRACES.items():
    if trace is None:
        continue
    post = trace.posterior
    rows.append({
        "model"  : name,
        "α mean" : float(post["alpha"].values.mean()),
        "α std"  : float(post["alpha"].values.std()),
        "λ mean" : float(post["lam"].values.mean()),
        "λ std"  : float(post["lam"].values.std()),
    })

pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
# ── R-hat 수렴 진단 ────────────────────────────────────────────────────────────
for name, trace in TRACES.items():
    if trace is None:
        continue
    rhat = az.rhat(trace)
    max_rhat = max(
        float(rhat[v].values.max())
        for v in ["phi", "alpha", "lam"]
        if v in rhat
    )
    status = "✓" if max_rhat < 1.01 else "△" if max_rhat < 1.05 else "✗"
    print(f"{status} {name:<22} max R-hat = {max_rhat:.4f}")